# I. FULL DATASET / 5 clusters (mcs 1900)

## 1. Decision tree for internal variables

In [27]:
# ── 0. Config ──────────────────────────────────────────────────────────────────

from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree, _tree
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import dtreeviz
import os

plt.style.use('default')

# ── Paths ─────────────────────────────────────────────────────────────────────
FULL_OUTPUT_DIR = "Results/Regular_clustering/Full_dataset"

run_label = "s2_balanced"
scaler    = "minmax"

FINAL_PARAMS = {
    #"s2_balanced_minmax": (3406, 34),
    "s2_balanced_minmax" : (2271, 15),
}

mcs, ms = FINAL_PARAMS[f"{run_label}_{scaler}"]

CSV_PATH = os.path.join(
    FULL_OUTPUT_DIR, "With_counts",
    scaler, run_label,
    f"final_mcs{mcs}_ms{ms}",
    f"clustering_{run_label}_{scaler}_mcs{mcs}_ms{ms}_labeled.csv"
)

OUT_DIR = os.path.join(
    FULL_OUTPUT_DIR, "With_counts",
    scaler, run_label,
    f"final_mcs{mcs}_ms{ms}",
    "decision_tree"
)

os.makedirs(OUT_DIR, exist_ok=True)
print(f"Input : {CSV_PATH}")
print(f"Output: {OUT_DIR}")

# ── Load dataset ───────────────────────────────────────────────────────────────

df = pd.read_csv(CSV_PATH, low_memory=False)

print(f"Dataset loaded : {len(df)} rows, {df.shape[1]} columns")

# ── 1. Feature definition ──────────────────────────────────────────────────────

IMAGING_COLS_BOOL = {
    "has_ultrasound", "has_ct_scan", "has_xray",
    "has_mri",
    #"has_radio_interventional", "has_nuclear_medicine",
}

BIO_EXAMS = {
    "has_blood_test", "has_culture",
    "has_lumbar_puncture", "has_blood_gas",
}

PROCEDURE_COLS   = {"had_ekg"}

DISPOSITION_COLS = {
    "hospitalization", "observation_unit",
    #"inter_facility_transfer",
}

COLS_QUANTI  = ["imaging_exam_count", "bio_exam_count"]
COLS_BOOL    = list(IMAGING_COLS_BOOL | BIO_EXAMS | PROCEDURE_COLS | DISPOSITION_COLS)
ALL_FEATURES = COLS_BOOL + COLS_QUANTI

#Ordre logique du plus au moins consommateur
#5 clusters
CLUSTER_LABELS_5 = {
    1:  "C1 — UHCD + Hospitalization + heavy workup",
    0:  "C2 — Hospitalized + full workup",
    2:  "C3 — Discharged + biology +/- ECG",
    4:  "C4 — Isolated X-ray +/- CT",
    3:  "C5 — Minimal consumption",
    -1: "Outliers",
}
CLUSTER_ORDER_5 = ["C1 — UHCD + Hospitalization + heavy workup",
                 "C2 — Hospitalized + full workup",
                 "C3 — Discharged + biology +/- ECG",
                 "C4 — Isolated X-ray +/- CT",
                 "C5 — Minimal consumption",
                 "Outliers"]


# 9 clusters
CLUSTER_LABELS_9 = {
    0:  "C1 — UHCD + Hospitalization + heavy workup",   # obs=1, hospi=1, bio=1.56, ekg=0.55
    4:  "C2 — Hospitalized + biology + imaging",         # hospi=1, bio=2.19, ct=0.74, ekg=0.34
    5:  "C3 — Hospitalized + biology",                   # hospi=1, bio=0.40, blood=0.20
    6:  "C4 — Hospitalized + ECG + CT",                  # hospi=1, ct=1.00, ekg=0.30, bio=1.00
    7:  "C5 — Hospitalized + ultrasound + biology",      # hospi=1, echo=0.98, bio=1.00
    8:  "C6 — Hospitalized + ECG",                       # hospi=1, ekg=0.48, bio=1.00
    1:  "C7 — Discharged + biology",                     # hospi=0, blood=1.00, bio=1.00
    3:  "C8 — Isolated X-ray",                           # xray=1.00, imaging=1.11
    2:  "C9 — Minimal consumption",                      # tout à 0
    -1: "Outliers",
}

CLUSTER_ORDER_9 = [
    "C1 — UHCD + Hospitalization + heavy workup",
    "C2 — Hospitalized + biology + imaging",
    "C3 — Hospitalized + biology",
    "C4 — Hospitalized + ECG + CT",
    "C5 — Hospitalized + ultrasound + biology",
    "C6 — Hospitalized + ECG",
    "C7 — Discharged + biology",
    "C8 — Isolated X-ray",
    "C9 — Minimal consumption",
    "Outliers",
]



FEATURE_RENAME = {
    'bp_status_ordinal':                  'Blood pressure',
    'hr_status_ordinal':                  'Heart rate',
    'temp_status_ordinal':                'Temperature',
    'sat_status_ordinal':                 'SpO2',
    'rr_status_ordinal':                  'Respiratory rate',
    'o2_flow_status_ordinal':             'O2 flow',
    'gcs_status_ordinal':                 'GCS',
    'cap_blood_sugar_status_ordinal':     'Blood sugar',
    'pupils_status_ordinal':              'Pupils size',
    'anisocoria_status_ordinal':          'Anisocoria',
    'urine_dipstick_clean_status_ordinal':'Urine dipstick',
    'pain_status_ordinal':                'Pain',
    'breathalyzer_status_ordinal':        'Breathalyzer',
    'hemocue_status_ordinal':             'Hemocue',
    'transport_ordinal':                  'Transport mode',
    'age':                                'Age',
    'triage':                             'Triage level',
    'sex':                                'Sex',
}

# ── 2. Prepare datasets ────────────────────────────────────────────────────────

df_model = df[ALL_FEATURES + ['cluster', 'cluster_label']].dropna()

# Tree 1 : clusters only (outliers removed)
df_clusters = df_model[df_model['cluster'] != -1]
X_clusters  = df_clusters[ALL_FEATURES]
y_clusters  = df_clusters['cluster_label']

# Tree 2 : outlier detection (everyone, binary target)
df_outliers               = df_model.copy()
df_outliers['is_outlier'] = (df_outliers['cluster'] == -1).astype(int)
X_outliers                = df_outliers[ALL_FEATURES]
y_outliers                = df_outliers['is_outlier']

print(f"\n── Tree 1 : cluster structure ──")
print(f"Individuals : {len(df_clusters)}")
print(f"Cluster distribution :\n{y_clusters.value_counts()}")



# ── 3. Helper : export tree image ─────────────────────────────────────────────

def export_tree_image(tree, feature_names, class_names, title, filename):
    fig, ax = plt.subplots(figsize=(32, 14), facecolor='white')
    ax.set_facecolor('white')
    plot_tree(
        tree,
        feature_names=feature_names,
        class_names=class_names,
        filled=True,
        rounded=True,
        fontsize=8,
        ax=ax
    )
    ax.set_title(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/{filename}.pdf", format="pdf", dpi=150,
                bbox_inches='tight', facecolor='white')
    plt.savefig(f"{OUT_DIR}/{filename}.png", format="png", dpi=200,
                bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Exported : {filename}.pdf and {filename}.png")

# ── 4. Helper : export feature importance ─────────────────────────────────────

def export_feature_importance(tree, feature_names, title, filename):
    importances = pd.Series(tree.feature_importances_, index=feature_names)
    importances = importances[importances > 0].sort_values(ascending=False)
    print(f"\nFeature importance ({title}) :")
    print(importances.round(3))
    fig, ax = plt.subplots(figsize=(8, 6), facecolor='white')
    ax.set_facecolor('white')
    importances.sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(title)
    ax.set_xlabel("Importance (Gini)")
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/{filename}.png", dpi=200,
                bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Exported : {filename}.png")

# ── 5. Helper : extract rules leading to a target class ───────────────────────

def get_target_rules(tree, feature_names, target_class):
    tree_      = tree.tree_
    classes    = tree.classes_
    feat_names = [
        feature_names[i] if i != _tree.TREE_UNDEFINED else "undefined"
        for i in tree_.feature
    ]
    rules = []

    def recurse(node, conditions):
        if tree_.feature[node] != _tree.TREE_UNDEFINED:
            name      = feat_names[node]
            threshold = tree_.threshold[node]
            recurse(tree_.children_left[node],  conditions + [f"{name} <= {threshold:.2f}"])
            recurse(tree_.children_right[node], conditions + [f"{name} >  {threshold:.2f}"])
        else:
            predicted_class = classes[np.argmax(tree_.value[node])]
            n_samples       = int(tree_.n_node_samples[node])
            purity          = float(np.max(tree_.value[node]) / n_samples)
            if predicted_class == target_class:
                rules.append({
                    'conditions': conditions,
                    'n_samples' : n_samples,
                    'purity'    : purity,
                })

    recurse(0, [])
    return rules

# ══════════════════════════════════════════════════════════════════════════════
# TREE 1 — Cluster structure (outliers excluded)
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("TREE 1 — Cluster structure")
print("="*60)

tree_clusters = DecisionTreeClassifier(
    max_depth=8,
    min_samples_leaf=300,
    random_state=42
)
tree_clusters.fit(X_clusters, y_clusters)

class_names_clusters = [f"Cluster {c}" for c in sorted(y_clusters.unique())]

export_tree_image(
    tree_clusters,
    ALL_FEATURES,
    class_names_clusters,
    title    = "Decision tree — Cluster structure (Scenario 2)",
    filename = "tree1_clusters"
)

export_feature_importance(
    tree_clusters,
    ALL_FEATURES,
    title    = "Most discriminating features — Cluster structure",
    filename = "tree1_feature_importance"
)

rules_clusters = export_text(tree_clusters, feature_names=ALL_FEATURES)
print("\nText rules :")
print(rules_clusters)

# ══════════════════════════════════════════════════════════════════════════════
# TREE 2 — Outlier detection (full dataset, binary target)
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("TREE 2 — Outlier detection")
print("="*60)

tree_outliers = DecisionTreeClassifier(
    max_depth=8,
    min_samples_leaf=40,
    random_state=42,
    class_weight='balanced'
)
tree_outliers.fit(X_outliers, y_outliers)

export_tree_image(
    tree_outliers,
    ALL_FEATURES,
    class_names = ["Non-outlier", "Outlier"],
    title       = "Decision tree — Outlier detection (Scenario 2)",
    filename    = "tree2_outliers"
)

export_feature_importance(
    tree_outliers,
    ALL_FEATURES,
    title    = "Most discriminating features — Outlier detection",
    filename = "tree2_feature_importance"
)

rules_outliers = export_text(tree_outliers, feature_names=ALL_FEATURES)
print("\nText rules :")
print(rules_outliers)

# ── Outlier paths ──────────────────────────────────────────────────────────────

outlier_rules = get_target_rules(tree_outliers, ALL_FEATURES, target_class=1)

print("\n── Paths leading to outlier leaves ──")
if not outlier_rules:
    print("No leaf predicts outliers as majority class.")
    print("Try reducing max_depth or min_samples_leaf.")
else:
    for r in outlier_rules:
        print(f"\n  n={r['n_samples']} individuals (purity {r['purity']:.0%})")
        for cond in r['conditions']:
            print(f"    {cond}")

Input : Results/Regular_clustering/Full_dataset/With_counts/minmax/s2_balanced/final_mcs2271_ms15/clustering_s2_balanced_minmax_mcs2271_ms15_labeled.csv
Output: Results/Regular_clustering/Full_dataset/With_counts/minmax/s2_balanced/final_mcs2271_ms15/decision_tree
Dataset loaded : 56781 rows, 194 columns

── Tree 1 : cluster structure ──
Individuals : 52580
Cluster distribution :
cluster_label
C9 — Minimal consumption                      12507
C1 — UHCD + Hospitalization + heavy workup    10306
C8 — Isolated X-ray                            5995
C6 — Hospitalized + ECG                        5306
C4 — Hospitalized + ECG + CT                   4924
C7 — Discharged + biology                      4286
C3 — Hospitalized + biology                    3636
C5 — Hospitalized + ultrasound + biology       3011
C2 — Hospitalized + biology + imaging          2609
Name: count, dtype: int64

TREE 1 — Cluster structure
Exported : tree1_clusters.pdf and tree1_clusters.png

Feature importance (Most di

# decision tree for external variables

In [28]:
#=============================
# building tree feature
#==============================




STATUS_ENCODINGS = {

    # ── Pression artérielle ───────────────────────────────────────────────────
    'bp_status': {
        'not_measured' : 1,
        'hypotension'  : 2,
        'normotension' : 3,
        'hypertension' : 4,
    },

    # ── Fréquence cardiaque ───────────────────────────────────────────────────
    'hr_status': {
        'not_measured' : 1,
        'bradycardia'  : 2,
        'normocardia'  : 3,
        'tachycardia'  : 4,
    },

    # ── Température ──────────────────────────────────────────────────────────
    'temp_status': {
        'not_measured' : 1,
        'hypothermia'  : 2,
        'normothermia' : 3,
        'hyperthermia' : 4,
    },

    # ── Saturation O2 ─────────────────────────────────────────────────────────
    'sat_status': {
        'not_measured' : 1,
        'severe hypoxia': 2,
        'hypoxia'      : 3,
        'normal'       : 4,
    },

    # ── Fréquence respiratoire ────────────────────────────────────────────────
    'rr_status': {
        'not_measured' : 1,
        'bradypnea'    : 2,
        'normal'       : 3,
        'tachypnea'    : 4,
    },

    # ── O2 flow — binaire ─────────────────────────────────────────────────────
    'o2_flow_status': {
        'not_measured' : 0,
        'off'          : 1,
        'on'           : 2,
    },

    # ── Glasgow ───────────────────────────────────────────────────────────────
    'gcs_status': {
        'not_measured'      : 1,
        'severe_impairment' : 2,
        'moderate_impairment': 3,
        'normal'            : 4,
    },

    # ── Glycémie capillaire ───────────────────────────────────────────────────
    'cap_blood_sugar_status': {
        'not_measured' : 1,
        'hypoglycemia' : 2,
        'normoglycemia': 3,
        'hyperglycemia': 4,
    },

    # ── Pupilles ──────────────────────────────────────────────────────────────
    'pupils_status': {
        'not_measured' : 1,
        'myosis'       : 2,
        'normal'       : 3,
        'mydriasis'    : 4,
    },

    # ── Anisocorie — binaire ──────────────────────────────────────────────────
    'anisocoria_status': {
        'not_measured' : 0,
        'no'           : 1,
        'yes'          : 2,
    },

    # ── Bandelette urinaire — binaire ─────────────────────────────────────────
    'urine_dipstick_clean_status': {
        'not_measured' : 0,
        'negative'     : 1,
        'positive'     : 2,
    },

    # ── Douleur ───────────────────────────────────────────────────────────────
    'pain_status': {
        'not_measured' : 1,
        'no_pain'      : 2,
        'mild_pain'    : 3,
        'moderate_pain': 4,
        'severe_pain'  : 5,
    },

    # ── Ethylotest — binaire ──────────────────────────────────────────────────
    'breathalyzer_status': {
        'not_measured' : 0,
        'negative'     : 1,
        'positive'     : 2,
    },

    # ── Hémocue ───────────────────────────────────────────────────────────────
    'hemocue_status': {
        'not_measured'  : 1,
        'severe_anemia' : 2,
        'moderate_anemia': 3,
        'normal'        : 4,
    },
}

# ── Application status encodings ──────────────────────────────────────────────
for col, mapping in STATUS_ENCODINGS.items():
    if col in df.columns:
        df[f'{col}_ordinal'] = df[col].map(mapping)
        n_nan = df[f'{col}_ordinal'].isna().sum()
        if n_nan > 0:
            print(f"⚠️  {col}: {n_nan} NaN non mappés")
        else:
            print(f"✅ {col}: encodé ({len(mapping)} modalités)")

# ── Transport ─────────────────────────────────────────────────────────────────
TRANSPORT_ORDER = {
    'Unknown'            : 1,
    'Personal'           : 2,
    'Post medical advice': 3,
    'Ambulance'          : 4,
    'Emergency services' : 5,
}
df['transport_ordinal'] = df['transport_grouped'].map(TRANSPORT_ORDER)
print("Distribution transport_ordinal :")
print(df['transport_ordinal'].value_counts().sort_index())
print(f"NaN restants : {df['transport_ordinal'].isna().sum()}")




✅ bp_status: encodé (4 modalités)
✅ hr_status: encodé (4 modalités)
✅ temp_status: encodé (4 modalités)
✅ sat_status: encodé (4 modalités)
✅ rr_status: encodé (4 modalités)
✅ o2_flow_status: encodé (3 modalités)
✅ gcs_status: encodé (4 modalités)
✅ cap_blood_sugar_status: encodé (4 modalités)
✅ pupils_status: encodé (4 modalités)
✅ anisocoria_status: encodé (3 modalités)
✅ urine_dipstick_clean_status: encodé (3 modalités)
✅ pain_status: encodé (5 modalités)
✅ breathalyzer_status: encodé (3 modalités)
✅ hemocue_status: encodé (4 modalités)
Distribution transport_ordinal :
transport_ordinal
1     2396
2    31546
3      305
4     7493
5    15041
Name: count, dtype: int64
NaN restants : 0


In [29]:
# ============================================================
# MODULAR CALLS — EXTERNAL DECISION TREES FOR 5 & 9 CLUSTERS
# ============================================================

CLUSTERING_RUNS = [
    {
        "cluster_solution": 5,
        "run_label": "s2_balanced",
        "scaler": "minmax",
        "mcs": 2271,
        "ms": 15,
        "cluster_labels": CLUSTER_LABELS_5,
        "cluster_order": CLUSTER_ORDER_5,
    },
    {
        "cluster_solution": 9,
        "run_label": "s2_balanced",
        "scaler": "minmax",
        "mcs": 3406,
        "ms": 34,
        "cluster_labels": CLUSTER_LABELS_9,
        "cluster_order": CLUSTER_ORDER_9,
    },
]


for config in CLUSTERING_RUNS:

    # ========================================================
    # CONFIG
    # ========================================================

    cluster_solution = config["cluster_solution"]
    run_label        = config["run_label"]
    scaler           = config["scaler"]
    mcs              = config["mcs"]
    ms               = config["ms"]
    CLUSTER_LABELS   = config["cluster_labels"]
    CLUSTER_ORDER    = config["cluster_order"]

    print("\n" + "=" * 80)
    print(f"RUNNING DECISION TREES — {cluster_solution} CLUSTERS")
    print("=" * 80)

    # ========================================================
    # PATHS
    # ========================================================

    CSV_PATH = os.path.join(
        FULL_OUTPUT_DIR,
        "With_counts",
        scaler,
        run_label,
        f"final_mcs{mcs}_ms{ms}",
        f"clustering_{run_label}_{scaler}_mcs{mcs}_ms{ms}_labeled.csv"
    )

    OUT_DIR_TREE = os.path.join(
        FULL_OUTPUT_DIR,
        "With_counts",
        scaler,
        run_label,
        f"final_mcs{mcs}_ms{ms}",
        f"decision_tree_external_{cluster_solution}clusters"
    )

    os.makedirs(OUT_DIR_TREE, exist_ok=True)

    print(f"Input : {CSV_PATH}")
    print(f"Output: {OUT_DIR_TREE}")

    # ========================================================
    # LOAD DATA
    # ========================================================

    df = pd.read_csv(CSV_PATH, low_memory=False)

    # ========================================================
    # ORDINAL LEGEND EXPORT
    # ========================================================

    fig, ax = plt.subplots(figsize=(10, 12), facecolor="white")
    ax.axis("off")

    table_data = [["Variable", "Value", "Label"]]

    for col, mapping in STATUS_ENCODINGS.items():
        col_clean = FEATURE_RENAME.get(
            f"{col}_ordinal",
            col.replace("_status", "").replace("_", " ")
        )

        for label, value in sorted(mapping.items(), key=lambda x: x[1]):
            table_data.append([
                col_clean,
                str(value),
                label.replace("_", " ")
            ])

    table = ax.table(
        cellText=table_data[1:],
        colLabels=table_data[0],
        cellLoc="left",
        loc="center",
        colWidths=[0.35, 0.1, 0.35],
    )

    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.4)

    for j in range(3):
        table[0, j].set_facecolor("#2c3e50")
        table[0, j].set_text_props(color="white", fontweight="bold")

    prev_var = None
    color_a = "#f5f5f5"
    color_b = "#ffffff"
    current_color = color_a

    for i in range(1, len(table_data)):
        var = table_data[i][0]
        if var != prev_var:
            current_color = color_b if current_color == color_a else color_a
            prev_var = var
        for j in range(3):
            table[i, j].set_facecolor(current_color)

    ax.set_title(
        f"Ordinal encoding legend — {cluster_solution}-cluster solution",
        fontsize=12,
        fontweight="bold",
        pad=20
    )

    plt.tight_layout()
    plt.savefig(
        os.path.join(OUT_DIR_TREE, "ordinal_legend.png"),
        dpi=200,
        bbox_inches="tight",
        facecolor="white"
    )
    plt.close()

    # ========================================================
    # PREPARE DATA
    # ========================================================

    cols_needed = EXTERNAL_FEATURES + ["cluster_label"]
    df_model = df[[c for c in cols_needed if c in df.columns]].dropna()

    # ========================================================
    # TREE 1 — MULTICLASS CLUSTERS
    # ========================================================

    df_clusters = df_model[df_model["cluster_label"] != "Outliers"].copy()

    X1 = df_clusters[EXTERNAL_FEATURES].astype(float)
    X1 = X1.rename(columns=FEATURE_RENAME)

    valid_classes = [c for c in CLUSTER_ORDER if c != "Outliers"]

    le1 = LabelEncoder()
    le1.classes_ = np.array(valid_classes)

    y1 = le1.transform(df_clusters["cluster_label"])
    class_names1 = list(le1.classes_)

    print(f"Tree 1 — N={len(X1)} | Classes: {class_names1}")

    tree1 = DecisionTreeClassifier(
        max_depth=4,
        min_samples_leaf=200,
        class_weight="balanced",
        random_state=42,
    )

    tree1.fit(X1, y1)

    viz1 = dtreeviz.model(
        tree1,
        X_train=X1,
        y_train=y1,
        feature_names=list(X1.columns),
        class_names=class_names1,
        target_name="Cluster",
    )

    v1 = viz1.view(fancy=True, scale=1.5, orientation="LR")
    v1.save(os.path.join(
        OUT_DIR_TREE,
        f"tree1_clusters_{cluster_solution}clusters.svg"
    ))

    v1_simple = viz1.view(fancy=False, scale=1.2, orientation="TD")
    v1_simple.save(os.path.join(
        OUT_DIR_TREE,
        f"tree1_clusters_{cluster_solution}clusters_simple.svg"
    ))

    # Feature importance
    imp1 = pd.Series(tree1.feature_importances_, index=X1.columns)
    imp1 = imp1[imp1 > 0].sort_values(ascending=False)

    fig, ax = plt.subplots(
        figsize=(9, max(5, len(imp1) * 0.4)),
        facecolor="white"
    )

    imp1.sort_values().plot(
        kind="barh",
        ax=ax,
        color="steelblue",
        edgecolor="white"
    )

    ax.axvline(
        imp1.mean(),
        color="red",
        linestyle="--",
        alpha=0.6,
        label="Mean"
    )

    ax.set_title(
        f"Feature Importance — Tree 1 ({cluster_solution} clusters)",
        fontsize=13,
        fontweight="bold"
    )

    ax.set_xlabel("Gini Importance")
    ax.legend()

    plt.tight_layout()
    plt.savefig(
        os.path.join(
            OUT_DIR_TREE,
            f"tree1_feature_importance_{cluster_solution}clusters.png"
        ),
        dpi=200,
        bbox_inches="tight"
    )
    plt.close()

    # Rules
    rules1 = export_text(tree1, feature_names=list(X1.columns))

    with open(
        os.path.join(
            OUT_DIR_TREE,
            f"tree1_rules_{cluster_solution}clusters.txt"
        ),
        "w"
    ) as f:
        f.write(rules1)

    # ========================================================
    # TREE 2 — BINARY OUTLIER DETECTION
    # ========================================================

    df_all = df_model.copy()

    X2 = df_all[EXTERNAL_FEATURES].astype(float)
    X2 = X2.rename(columns=FEATURE_RENAME)

    y2 = (df_all["cluster_label"] == "Outliers").astype(int)

    class_names2 = ["Clustered", "Outlier"]

    print(
        f"Tree 2 — N={len(X2)} | "
        f"Outliers: {y2.sum()} ({y2.mean():.1%})"
    )

    tree2 = DecisionTreeClassifier(
        max_depth=4,
        min_samples_leaf=200,
        class_weight="balanced",
        random_state=42,
    )

    tree2.fit(X2, y2)

    viz2 = dtreeviz.model(
        tree2,
        X_train=X2,
        y_train=y2,
        feature_names=list(X2.columns),
        class_names=class_names2,
        target_name="Outlier",
    )

    v2 = viz2.view(fancy=True, scale=1.5, orientation="LR")
    v2.save(os.path.join(
        OUT_DIR_TREE,
        f"tree2_outliers_{cluster_solution}clusters.svg"
    ))

    v2_simple = viz2.view(fancy=False, scale=1.2, orientation="TD")
    v2_simple.save(os.path.join(
        OUT_DIR_TREE,
        f"tree2_outliers_{cluster_solution}clusters_simple.svg"
    ))

    # Feature importance
    imp2 = pd.Series(tree2.feature_importances_, index=X2.columns)
    imp2 = imp2[imp2 > 0].sort_values(ascending=False)

    fig, ax = plt.subplots(
        figsize=(9, max(5, len(imp2) * 0.4)),
        facecolor="white"
    )

    imp2.sort_values().plot(
        kind="barh",
        ax=ax,
        color="tomato",
        edgecolor="white"
    )

    ax.axvline(
        imp2.mean(),
        color="red",
        linestyle="--",
        alpha=0.6,
        label="Mean"
    )

    ax.set_title(
        f"Feature Importance — Tree 2 ({cluster_solution} clusters outliers)",
        fontsize=13,
        fontweight="bold"
    )

    ax.set_xlabel("Gini Importance")
    ax.legend()

    plt.tight_layout()
    plt.savefig(
        os.path.join(
            OUT_DIR_TREE,
            f"tree2_feature_importance_{cluster_solution}clusters.png"
        ),
        dpi=200,
        bbox_inches="tight"
    )
    plt.close()

    # Rules
    rules2 = export_text(tree2, feature_names=list(X2.columns))

    with open(
        os.path.join(
            OUT_DIR_TREE,
            f"tree2_rules_{cluster_solution}clusters.txt"
        ),
        "w"
    ) as f:
        f.write(rules2)

    print(f"Completed {cluster_solution}-cluster analysis.")


RUNNING DECISION TREES — 5 CLUSTERS
Input : Results/Regular_clustering/Full_dataset/With_counts/minmax/s2_balanced/final_mcs2271_ms15/clustering_s2_balanced_minmax_mcs2271_ms15_labeled.csv
Output: Results/Regular_clustering/Full_dataset/With_counts/minmax/s2_balanced/final_mcs2271_ms15/decision_tree_external_5clusters


KeyError: "['transport_ordinal', 'bp_status_ordinal', 'hr_status_ordinal', 'temp_status_ordinal', 'sat_status_ordinal', 'rr_status_ordinal', 'o2_flow_status_ordinal', 'gcs_status_ordinal', 'cap_blood_sugar_status_ordinal', 'pupils_status_ordinal', 'anisocoria_status_ordinal', 'urine_dipstick_clean_status_ordinal', 'pain_status_ordinal', 'breathalyzer_status_ordinal', 'hemocue_status_ordinal'] not in index"

In [ ]:
# # ========================================
# # CALL
# #=========================================
#
#
#
# # ── Features ──────────────────────────────────────────────────────────────────
# STATUS_ORDINAL_FEATURES = [
#     'bp_status_ordinal', 'hr_status_ordinal', 'temp_status_ordinal',
#     'sat_status_ordinal', 'rr_status_ordinal', 'o2_flow_status_ordinal',
#     'gcs_status_ordinal', 'cap_blood_sugar_status_ordinal',
#     'pupils_status_ordinal', 'anisocoria_status_ordinal',
#     'urine_dipstick_clean_status_ordinal', 'pain_status_ordinal',
#     'breathalyzer_status_ordinal', 'hemocue_status_ordinal',
# ]
#
# EXTERNAL_FEATURES = (
#     ["age", "triage", "transport_ordinal", "sex_bin"]
#     + STATUS_ORDINAL_FEATURES
# )
#
# OUT_DIR_TREE = os.path.join(
#     FULL_OUTPUT_DIR, "With_counts", scaler, run_label,
#     f"final_mcs{mcs}_ms{ms}", "decision_tree_external"
# )
# os.makedirs(OUT_DIR_TREE, exist_ok=True)
#
#
# # ── Ordinal legend table ──────────────────────────────────────────────────────
# fig, ax = plt.subplots(figsize=(10, 12), facecolor="white")
# ax.axis("off")
#
# # Construire les données du tableau
# table_data = [["Variable", "Value", "Label"]]
# for col, mapping in STATUS_ENCODINGS.items():
#     col_clean = FEATURE_RENAME.get(f"{col}_ordinal", col.replace("_status", "").replace("_", " "))
#     for label, value in sorted(mapping.items(), key=lambda x: x[1]):
#         table_data.append([col_clean, str(value), label.replace("_", " ")])
#
# table = ax.table(
#     cellText  = table_data[1:],
#     colLabels = table_data[0],
#     cellLoc   = "left",
#     loc       = "center",
#     colWidths = [0.35, 0.1, 0.35],
# )
#
# table.auto_set_font_size(False)
# table.set_fontsize(9)
# table.scale(1, 1.4)
#
# # Style header
# for j in range(3):
#     table[0, j].set_facecolor("#2c3e50")
#     table[0, j].set_text_props(color="white", fontweight="bold")
#
# # Alterner les couleurs de lignes + regrouper par variable
# prev_var = None
# color_a  = "#f5f5f5"
# color_b  = "#ffffff"
# current_color = color_a
#
# for i in range(1, len(table_data)):
#     var = table_data[i][0]
#     if var != prev_var:
#         current_color = color_b if current_color == color_a else color_a
#         prev_var = var
#     for j in range(3):
#         table[i, j].set_facecolor(current_color)
#
# ax.set_title("Ordinal encoding legend — decision tree features",
#              fontsize=12, fontweight="bold", pad=20)
# plt.tight_layout()
# plt.savefig(os.path.join(OUT_DIR_TREE, "ordinal_legend.png"),
#             dpi=200, bbox_inches="tight", facecolor="white")
# plt.close()
# print("Saved: ordinal_legend.png")
#
#
#
# # ══════════════════════════════════════════════════════════════════════════════
# # TREE 1 — Cluster structure (outliers excluded)
# # ══════════════════════════════════════════════════════════════════════════════
#
# cols_needed  = EXTERNAL_FEATURES + ["cluster_label"]
# df_model     = df[[c for c in cols_needed if c in df.columns]].dropna()
# df_clusters  = df_model[df_model["cluster_label"] != "Outliers"].copy()
#
# X1 = df_clusters[EXTERNAL_FEATURES].astype(float)
# X1 = X1.rename(columns=FEATURE_RENAME)
# y1_raw = pd.Categorical(
#     df_clusters["cluster_label"],
#     categories=CLUSTER_ORDER,
#     ordered=True
# )
#
# # Encode en entiers en respectant CLUSTER_ORDER
# from sklearn.preprocessing import LabelEncoder
# le1          = LabelEncoder()
# le1.classes_ = np.array([c for c in CLUSTER_ORDER if c != "Outliers"])
# y1           = le1.transform(df_clusters["cluster_label"])
# class_names1 = list(le1.classes_)
#
# print(f"Tree 1 — N={len(X1)} | Classes: {class_names1}")
#
# tree1 = DecisionTreeClassifier(
#     max_depth        = 4,
#     min_samples_leaf = 200,
#     class_weight     = "balanced",
#     random_state     = 42,
# )
# tree1.fit(X1, y1)
#
# # dtreeviz
# viz1 = dtreeviz.model(
#     tree1,
#     X_train       = X1,
#     y_train       = y1,
#     feature_names = list(X1.columns),
#     class_names   = class_names1,
#     target_name   = "Cluster",
# )
# v1 = viz1.view(fancy=True, scale=1.5, orientation="LR")
# v1.save(os.path.join(OUT_DIR_TREE, "tree1_clusters.svg"))
# #v1.save(os.path.join(OUT_DIR_TREE, "tree1_clusters.pdf"))
#
# v1_simple = viz1.view(fancy=False, scale=1.2, orientation="TD")
# v1_simple.save(os.path.join(OUT_DIR_TREE, "tree1_clusters_simple.svg"))
# print("Tree 1 saved")
#
# # Feature importance
# imp1 = pd.Series(tree1.feature_importances_, index=X1.columns)
# imp1 = imp1[imp1 > 0].sort_values(ascending=False)
# fig, ax = plt.subplots(figsize=(9, max(5, len(imp1) * 0.4)), facecolor="white")
# imp1.sort_values().plot(kind="barh", ax=ax, color="steelblue", edgecolor="white")
# ax.axvline(imp1.mean(), color="red", linestyle="--", alpha=0.6, label="Mean")
# ax.set_title("Feature Importance — Tree 1 (cluster structure)", fontsize=13, fontweight="bold")
# ax.set_xlabel("Gini Importance")
# ax.legend()
# plt.tight_layout()
# plt.savefig(os.path.join(OUT_DIR_TREE, "tree1_feature_importance.png"), dpi=200, bbox_inches="tight")
# plt.close()
#
# # ══════════════════════════════════════════════════════════════════════════════
# # TREE 2 — Outlier detection (binary: clustered vs outlier)
# # ══════════════════════════════════════════════════════════════════════════════
#
# df_all  = df_model.copy()
# X2      = df_all[EXTERNAL_FEATURES].astype(float)
# X2      = X2.rename(columns=FEATURE_RENAME)
# y2      = (df_all["cluster_label"] == "Outliers").astype(int)
# class_names2 = ["Clustered", "Outlier"]
#
# print(f"\nTree 2 — N={len(X2)} | Outliers: {y2.sum()} ({y2.mean():.1%})")
#
# tree2 = DecisionTreeClassifier(
#     max_depth        = 4,
#     min_samples_leaf = 200,
#     class_weight     = "balanced",
#     random_state     = 42,
# )
# tree2.fit(X2, y2)
#
# viz2 = dtreeviz.model(
#     tree2,
#     X_train       = X2,
#     y_train       = y2,
#     feature_names = list(X2.columns),
#     class_names   = class_names2,
#     target_name   = "Outlier",
# )
# v2 = viz2.view(fancy=True, scale=1.5, orientation="LR")
# v2.save(os.path.join(OUT_DIR_TREE, "tree2_outliers.svg"))
# #v2.save(os.path.join(OUT_DIR_TREE, "tree2_outliers.pdf"))
#
# v2_simple = viz2.view(fancy=False, scale=1.2, orientation="TD")
# v2_simple.save(os.path.join(OUT_DIR_TREE, "tree2_outliers_simple.svg"))
# print("Tree 2 saved")
#
# # Feature importance
# imp2 = pd.Series(tree2.feature_importances_, index=X2.columns)
# imp2 = imp2[imp2 > 0].sort_values(ascending=False)
# fig, ax = plt.subplots(figsize=(9, max(5, len(imp2) * 0.4)), facecolor="white")
# imp2.sort_values().plot(kind="barh", ax=ax, color="tomato", edgecolor="white")
# ax.axvline(imp2.mean(), color="red", linestyle="--", alpha=0.6, label="Mean")
# ax.set_title("Feature Importance — Tree 2 (outlier detection)", fontsize=13, fontweight="bold")
# ax.set_xlabel("Gini Importance")
# ax.legend()
# plt.tight_layout()
# plt.savefig(os.path.join(OUT_DIR_TREE, "tree2_feature_importance.png"), dpi=200, bbox_inches="tight")
# plt.close()
#
# # ── Tree 1 text rules ─────────────────────────────────────────────────────────
# from sklearn.tree import export_text
# rules1 = export_text(tree1, feature_names=list(X1.columns))
# print(rules1)
# with open(os.path.join(OUT_DIR_TREE, "tree1_rules.txt"), "w") as f:
#     f.write(rules1)
# print("Saved: tree1_rules.txt")
#
# # ── Tree 2 text rules ─────────────────────────────────────────────────────────
# rules2 = export_text(tree2, feature_names=list(X2.columns))
# print(rules2)
# with open(os.path.join(OUT_DIR_TREE, "tree2_rules.txt"), "w") as f:
#     f.write(rules2)
# print("Saved: tree2_rules.txt")